---
title: "Final Project: Regression"
format: 
  html:
    embed-resources: true
execute:
  echo: true
code-fold: true
author: James Compagno
jupyter: python3
---

In [1]:
import numpy as np
import pandas as pd
import plotnine as p9
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Citation/Attribution: Claude and ChatGPT were used to adapt existing code to make this project. They were also used to create the log_rmse

# **Data Fields**

Below is a brief description of all variables in the dataset:

**SalePrice** – Property sale price in dollars (target variable)

**PID** – Unique property ID number

**LotFrontage** – Linear feet of street connected to property

**LotArea** – Lot size in square feet

**Street** – Type of road access

**Neighborhood** – Physical location within Ames city limits

**BldgType** – Type of dwelling

**HouseStyle** – Style of dwelling

**OverallQual** – Overall material and finish quality

**OverallCond** – Overall condition rating

**YearBuilt** – Original construction year

**RoofStyle** – Type of roof

**Heating** – Type of heating system

**CentralAir** – Whether the house has central air conditioning

**Electrical** – Electrical system type

**GrLivArea** – Above-grade living area (sq ft)

**FullBath** – Full bathrooms above grade

**HalfBath** – Half bathrooms above grade

**Bedroom AbvGr** – Number of bedrooms above basement level

**TotRmsAbvGrd** – Total rooms above grade (excluding bathrooms)

**Functional** – Home functionality rating

**ScreenPorch** – Screen porch area (sq ft)

**PoolArea** – Pool area (sq ft)

**YrSold** – Year sold

**SaleType** – Type of sale

In [2]:
# Read the data
ames_train = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-regression/train_new.csv")

# Load data and prepare features/target
X = ames_train.drop(["SalePrice", "PID"], axis=1)
# y = ames_train["SalePrice"]
y = np.log1p(ames_train["SalePrice"])

In [3]:
# Model Library 
model_library = {}
records = []

In [4]:
# Read test data
ames_test = pd.read_csv("/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-regression/test_new.csv")
test_PID = ames_test["PID"]
X_test = ames_test.drop("PID", axis=1)

In [5]:
ct = ColumnTransformer(
    [
        ("num", 
         Pipeline([
             ("impute", SimpleImputer(strategy="median")),
             ("scale", StandardScaler())
         ]), 
         make_column_selector(dtype_include=np.number)),
        ("cat", 
         Pipeline([
             ("impute", SimpleImputer(strategy="most_frequent")),
             ("encode", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
         ]), 
         make_column_selector(dtype_include=object))
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# NOTE 

Added imputation

First fills missing values with the median of that column

Then standardizes (mean=0, std=1)

Median is robust to outliers (better than mean for housing prices which can be skewed)

For categorical columns:

First fills missing values with the most common category in that column

Then one-hot encodes

In [6]:
from sklearn.metrics import make_scorer

def log_rmse(y_true, y_pred):
    # Clip predictions to avoid log(0) errors
    y_pred = np.clip(y_pred, 1, None)
    return np.sqrt(mean_squared_error(np.log(y_true), np.log(y_pred)))

# Make it a scorer (greater_is_better=False because lower RMSE is better)
log_rmse_scorer = make_scorer(log_rmse, greater_is_better=False)

# Linear Regression

In [7]:
def run_linear_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Cross Validation Pipeline
    pipe = Pipeline([
        ("preprocess", ct),
        ("linear_regression", LinearRegression())
    ])
    
    # Fit and add to Library
    pipe.fit(X_subset, y)
    model_library[model_name] = pipe
    
    # Get top 10 coefficients
    feature_names = pipe.named_steps['preprocess'].get_feature_names_out()
    coefficients = pipe.named_steps['linear_regression'].coef_
    
    coef_df = pd.DataFrame({
        'Variable': feature_names,
        'Coefficient': coefficients
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Metrics Calculation 
    rmse = cross_val_score(pipe, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(pipe, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(pipe, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(pipe, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "Linear",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),  
        "Hyperparameter 1 Name": "NA", 
        "Hyperparameter 1 Value": "NA",
        "Hyperparameter 2 Name": "NA", 
        "Hyperparameter 2 Value": "NA",
        "Hyperparameter 3 Name": "NA", 
        "Hyperparameter 3 Value": "NA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models


In [8]:
run_linear_regression("All_Features_Linear", None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."


# Ridge

In [9]:
def run_ridge_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """

    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Pipeline with Ridge
    pipe = Pipeline([
        ("preprocess", ct),
        ("ridge", Ridge())
    ])
    
    # GridSearchCV - tune on log_rmse_scorer for best Kaggle performance
    param_grid = {'ridge__alpha': np.logspace(-1, 2, 50)}
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best alpha
    best_alpha = gscv.best_params_['ridge__alpha']
    
    # Get top 10 coefficients
    coef_df = pd.DataFrame({
        'Variable': gscv.best_estimator_.named_steps['preprocess'].get_feature_names_out(),
        'Coefficient': gscv.best_estimator_.named_steps['ridge'].coef_
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Store coefficients in model library
    model_library[f"{model_name}_coefficients"] = coef_df
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)  # FIXED: was pipe, now gscv.best_estimator_
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "Ridge",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),  
        "Hyperparameter 1 Name": "Alpha", 
        "Hyperparameter 1 Value": best_alpha,
        "Hyperparameter 2 Name": "NA", 
        "Hyperparameter 2 Value": "NA",
        "Hyperparameter 3 Name": "NA", 
        "Hyperparameter 3 Value": "NA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [10]:
run_ridge_regression("Ridge_All_Tuned", None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."


# Lasso

In [11]:
def run_lasso_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Pipeline with Lasso
    pipe = Pipeline([
        ("preprocess", ct),
        ("lasso", Lasso())
    ])
    
    # GridSearchCV to tune alpha - using log_rmse_scorer for Kaggle metric
    param_grid = {'lasso__alpha': np.logspace(-1, 2, 50)}
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best alpha
    best_alpha = gscv.best_params_['lasso__alpha']
    
    # Get top 10 coefficients
    coef_df = pd.DataFrame({
        'Variable': gscv.best_estimator_.named_steps['preprocess'].get_feature_names_out(),
        'Coefficient': gscv.best_estimator_.named_steps['lasso'].coef_
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Store coefficients in model library
    model_library[f"{model_name}_coefficients"] = coef_df
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "Lasso",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "Alpha", 
        "Hyperparameter 1 Value": best_alpha,
        "Hyperparameter 2 Name": "NA", 
        "Hyperparameter 2 Value": "NA",
        "Hyperparameter 3 Name": "NA", 
        "Hyperparameter 3 Value": "NA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [12]:
run_lasso_regression("Lasso_All_Tuned", None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."


# Elastic Net

In [13]:
def run_elasticnet_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Pipeline with ElasticNet
    pipe = Pipeline([
        ("preprocess", ct),
        ("elasticnet", ElasticNet())
    ])
    
    # GridSearchCV to tune alpha and l1_ratio - using log_rmse_scorer for Kaggle metric
    param_grid = {
        'elasticnet__alpha': np.logspace(-1, 2, 50),
        'elasticnet__l1_ratio': [0, 0.25, 0.5, 0.75, 1]
    }
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best hyperparameters
    best_alpha = gscv.best_params_['elasticnet__alpha']
    best_l1_ratio = gscv.best_params_['elasticnet__l1_ratio']
    
    # Get top 10 coefficients
    coef_df = pd.DataFrame({
        'Variable': gscv.best_estimator_.named_steps['preprocess'].get_feature_names_out(),
        'Coefficient': gscv.best_estimator_.named_steps['elasticnet'].coef_
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Store coefficients in model library
    model_library[f"{model_name}_coefficients"] = coef_df
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "ElasticNet",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "Alpha", 
        "Hyperparameter 1 Value": best_alpha,
        "Hyperparameter 2 Name": "L1 Ratio", 
        "Hyperparameter 2 Value": best_l1_ratio,
        "Hyperparameter 3 Name": "NA", 
        "Hyperparameter 3 Value": "NA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [14]:
run_elasticnet_regression("ElasticNet_All_Tuned",None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."


# KNN

In [15]:
def run_knn_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Pipeline with KNN
    pipe = Pipeline([
        ("preprocess", ct),
        ("knn", KNeighborsRegressor())
    ])
    
    # GridSearchCV to tune n_neighbors, weights, and metric - using log_rmse_scorer for Kaggle metric
    param_grid = {
        'knn__n_neighbors': list(range(3, 25, 2)),
        'knn__weights': ['uniform', 'distance'],
        'knn__metric': ['euclidean', 'manhattan']
    }
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best hyperparameters
    best_n_neighbors = gscv.best_params_['knn__n_neighbors']
    best_weights = gscv.best_params_['knn__weights']
    best_metric = gscv.best_params_['knn__metric']
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "KNN",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "n_neighbors", 
        "Hyperparameter 1 Value": best_n_neighbors,
        "Hyperparameter 2 Name": "weights", 
        "Hyperparameter 2 Value": best_weights,
        "Hyperparameter 3 Name": "metric", 
        "Hyperparameter 3 Value": best_metric,
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": "NA (KNN has no coefficients)",
        "Top 10 Coefficients": "NA (KNN has no coefficients)"
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [16]:
run_knn_regression("KNN_All_Tuned", None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."
4,KNN_All_Tuned,KNN,CV-5,0.165188,0.027344,0.837218,0.013945,n_neighbors,7,weights,distance,metric,manhattan,All,NA (KNN has no coefficients),NA (KNN has no coefficients)


# Decision Tree

In [17]:
def run_decision_tree_regression(model_name, features=None):
    """
    model_name - will be stored in model_library
    features - list or None
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Pipeline with Decision Tree
    pipe = Pipeline([
        ("preprocess", ct),
        ("dt", DecisionTreeRegressor(random_state=42))
    ])
    
    # GridSearchCV to tune max_depth, min_samples_leaf, min_samples_split - using log_rmse_scorer for Kaggle metric
    param_grid = {
        'dt__max_depth': [5, 10, 15, 20, None],
        'dt__min_samples_leaf': [1, 2, 5, 10, 20],
        'dt__min_samples_split': [2, 5, 10, 20]
    }
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best hyperparameters
    best_max_depth = gscv.best_params_['dt__max_depth']
    best_min_samples_leaf = gscv.best_params_['dt__min_samples_leaf']
    best_min_samples_split = gscv.best_params_['dt__min_samples_split']
    
    # Get feature importances (Decision Tree's version of "important variables")
    feature_names = gscv.best_estimator_.named_steps['preprocess'].get_feature_names_out()
    importances = gscv.best_estimator_.named_steps['dt'].feature_importances_
    
    importance_df = pd.DataFrame({
        'Variable': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False).head(10)
    
    # Store importances in model library
    model_library[f"{model_name}_importances"] = importance_df
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": "Decision Tree",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "max_depth", 
        "Hyperparameter 1 Value": best_max_depth,
        "Hyperparameter 2 Name": "min_samples_leaf", 
        "Hyperparameter 2 Value": best_min_samples_leaf,
        "Hyperparameter 3 Name": "min_samples_split", 
        "Hyperparameter 3 Value": best_min_samples_split,
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(importance_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.4f}" for c in importance_df['Importance'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [18]:
run_decision_tree_regression("DecisionTree_All_Tuned", None)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."
4,KNN_All_Tuned,KNN,CV-5,0.165188,0.027344,0.837218,0.013945,n_neighbors,7,weights,distance,metric,manhattan,All,NA (KNN has no coefficients),NA (KNN has no coefficients)
5,DecisionTree_All_Tuned,Decision Tree,CV-5,0.178749,0.032041,0.809408,0.015043,max_depth,15,min_samples_leaf,2,min_samples_split,20,All,"Overall Qual, Gr Liv Area, Year Built, Lot Are...","0.7177, 0.1481, 0.0545, 0.0339, 0.0109, 0.0049..."


In [19]:
def run_ridge_poly_regression(model_name, features=None, degree=2, interaction_only=False):
    """
    model_name - will be stored in model_library
    features - list or None
    degree - polynomial degree (default 2)
    interaction_only - if True, only interaction terms (no x^2), if False, full polynomial
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Column Transformer with Polynomial Features for numeric columns
    ct_poly = ColumnTransformer(
        [
            ("num", 
             Pipeline([
                 ("impute", SimpleImputer(strategy="median")),
                 ("scale", StandardScaler()),
                 ("poly", PolynomialFeatures(degree=degree, interaction_only=interaction_only, include_bias=False))
             ]), 
             make_column_selector(dtype_include=np.number)),
            ("cat", 
             Pipeline([
                 ("impute", SimpleImputer(strategy="most_frequent")),
                 ("encode", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
             ]), 
             make_column_selector(dtype_include=object))
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")
    
    # Pipeline with Ridge
    pipe = Pipeline([
        ("preprocess", ct_poly),
        ("ridge", Ridge())
    ])
    
    # GridSearchCV to tune alpha - using log_rmse_scorer for Kaggle metric
    param_grid = {
        'ridge__alpha': np.logspace(-2, 3, 50)
    }
    gscv = GridSearchCV(pipe, param_grid, cv=5, scoring=log_rmse_scorer)
    gscv.fit(X_subset, y)
    
    # Add best model to Library
    model_library[model_name] = gscv.best_estimator_
    
    # Get best alpha
    best_alpha = gscv.best_params_['ridge__alpha']
    
    # Get top 10 coefficients
    feature_names = gscv.best_estimator_.named_steps['preprocess'].get_feature_names_out()
    coefficients = gscv.best_estimator_.named_steps['ridge'].coef_
    
    coef_df = pd.DataFrame({
        'Variable': feature_names,
        'Coefficient': coefficients
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Store coefficients in model library
    model_library[f"{model_name}_coefficients"] = coef_df
    
    # Metrics Calculation using best model
    rmse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(gscv.best_estimator_, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Determine poly type for recording
    poly_type = "interact_only" if interaction_only else "full"
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": f"Ridge_Poly_{poly_type}",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "Alpha", 
        "Hyperparameter 1 Value": best_alpha,
        "Hyperparameter 2 Name": "degree", 
        "Hyperparameter 2 Value": degree,
        "Hyperparameter 3 Name": "interaction_only", 
        "Hyperparameter 3 Value": interaction_only,
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [20]:
# Full polynomial (x², y², xy terms)
run_ridge_poly_regression("Ridge_Poly_Full", None, degree=2, interaction_only=False)

# Interaction only (xy terms, no squared)
run_ridge_poly_regression("Ridge_Poly_Interact", None, degree=2, interaction_only=True)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."
4,KNN_All_Tuned,KNN,CV-5,0.165188,0.027344,0.837218,0.013945,n_neighbors,7,weights,distance,metric,manhattan,All,NA (KNN has no coefficients),NA (KNN has no coefficients)
5,DecisionTree_All_Tuned,Decision Tree,CV-5,0.178749,0.032041,0.809408,0.015043,max_depth,15,min_samples_leaf,2,min_samples_split,20,All,"Overall Qual, Gr Liv Area, Year Built, Lot Are...","0.7177, 0.1481, 0.0545, 0.0339, 0.0109, 0.0049..."
6,Ridge_Poly_Full,Ridge_Poly_full,CV-5,0.140536,0.019807,0.881016,0.011925,Alpha,2.811769,degree,2,interaction_only,False,All,"Functional_Sal, Gr Liv Area, Neighborhood_Mead...","-0.35, 0.22, -0.13, 0.13, -0.12, -0.12, 0.12, ..."
7,Ridge_Poly_Interact,Ridge_Poly_interact_only,CV-5,0.140526,0.019797,0.881043,0.011904,Alpha,7.196857,degree,2,interaction_only,True,All,"Gr Liv Area, Functional_Sal, Bldg Type_Duplex,...","0.21, -0.17, -0.12, 0.11, 0.11, 0.11, -0.11, -..."


In [21]:
def run_poly_regression(model_name, features=None, degree=2, interaction_only=False):
    """
    model_name - will be stored in model_library
    features - list or None
    degree - polynomial degree (default 2)
    interaction_only - if True, only interaction terms (no x^2), if False, full polynomial
    Returns: DataFrame 
    """
    
    # Select features
    if features is not None:
        X_subset = X[features]
    else:
        X_subset = X
    
    # Column Transformer with Polynomial Features for numeric columns
    ct_poly = ColumnTransformer(
        [
            ("num", 
             Pipeline([
                 ("impute", SimpleImputer(strategy="median")),
                 ("scale", StandardScaler()),
                 ("poly", PolynomialFeatures(degree=degree, interaction_only=interaction_only, include_bias=False))
             ]), 
             make_column_selector(dtype_include=np.number)),
            ("cat", 
             Pipeline([
                 ("impute", SimpleImputer(strategy="most_frequent")),
                 ("encode", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
             ]), 
             make_column_selector(dtype_include=object))
        ],
        verbose_feature_names_out=False,
    ).set_output(transform="pandas")
    
    # Pipeline with Linear Regression
    pipe = Pipeline([
        ("preprocess", ct_poly),
        ("linear_regression", LinearRegression())
    ])
    
    # Fit and add to Library
    pipe.fit(X_subset, y)
    model_library[model_name] = pipe
    
    # Get top 10 coefficients
    feature_names = pipe.named_steps['preprocess'].get_feature_names_out()
    coefficients = pipe.named_steps['linear_regression'].coef_
    
    coef_df = pd.DataFrame({
        'Variable': feature_names,
        'Coefficient': coefficients
    }).sort_values('Coefficient', key=abs, ascending=False).head(10)
    
    # Store coefficients in model library
    model_library[f"{model_name}_coefficients"] = coef_df
    
    # Metrics Calculation
    rmse = cross_val_score(pipe, X_subset, y, cv=5, scoring='neg_root_mean_squared_error')
    mse = cross_val_score(pipe, X_subset, y, cv=5, scoring='neg_mean_squared_error')
    r2 = cross_val_score(pipe, X_subset, y, cv=5, scoring='r2')
    log_rmse_cv = cross_val_score(pipe, X_subset, y, cv=5, scoring=log_rmse_scorer)
    
    # Determine poly type for recording
    poly_type = "interact_only" if interaction_only else "full"
    
    # Metrics Storage 
    records.append({
        "Model": model_name,
        "Regression Type": f"Poly_Linear_{poly_type}",
        "Split": "CV-5",
        "RMSE Mean": -rmse.mean(),
        "MSE Mean": -mse.mean(),
        "R2 Mean": r2.mean(),
        "Log RMSE Mean": -log_rmse_cv.mean(),
        "Hyperparameter 1 Name": "degree", 
        "Hyperparameter 1 Value": degree,
        "Hyperparameter 2 Name": "interaction_only", 
        "Hyperparameter 2 Value": interaction_only,
        "Hyperparameter 3 Name": "NA", 
        "Hyperparameter 3 Value": "NA",
        "Variables Used": len(X_subset.columns) if features else "All",
        "Top 10 Variables": ", ".join(coef_df['Variable'].head(10).tolist()),
        "Top 10 Coefficients": ", ".join([f"{c:.2f}" for c in coef_df['Coefficient'].head(10).tolist()])
    })
    
    # Display
    cumulative_models = pd.DataFrame(records)
    return cumulative_models

In [22]:
# Full polynomial (x², y², xy terms)
run_poly_regression("Poly_Full_Linear", None, degree=2, interaction_only=False)

# Interaction only (xy terms, no squared)
run_poly_regression("Poly_Interact_Linear", None, degree=2, interaction_only=True)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."
4,KNN_All_Tuned,KNN,CV-5,0.165188,0.027344,0.837218,0.013945,n_neighbors,7,weights,distance,metric,manhattan,All,NA (KNN has no coefficients),NA (KNN has no coefficients)
5,DecisionTree_All_Tuned,Decision Tree,CV-5,0.178749,0.032041,0.809408,0.015043,max_depth,15,min_samples_leaf,2,min_samples_split,20,All,"Overall Qual, Gr Liv Area, Year Built, Lot Are...","0.7177, 0.1481, 0.0545, 0.0339, 0.0109, 0.0049..."
6,Ridge_Poly_Full,Ridge_Poly_full,CV-5,0.140536,0.019807,0.881016,0.011925,Alpha,2.811769,degree,2,interaction_only,False,All,"Functional_Sal, Gr Liv Area, Neighborhood_Mead...","-0.35, 0.22, -0.13, 0.13, -0.12, -0.12, 0.12, ..."
7,Ridge_Poly_Interact,Ridge_Poly_interact_only,CV-5,0.140526,0.019797,0.881043,0.011904,Alpha,7.196857,degree,2,interaction_only,True,All,"Gr Liv Area, Functional_Sal, Bldg Type_Duplex,...","0.21, -0.17, -0.12, 0.11, 0.11, 0.11, -0.11, -..."
8,Poly_Full_Linear,Poly_Linear_full,CV-5,0.142488,0.020382,0.877540,0.012086,degree,2,interaction_only,False,NA,NA,All,"Functional_Sal, Heating_Wall, Neighborhood_Grn...","-1.74, 0.30, 0.29, 0.25, 0.23, -0.21, 0.21, 0...."
9,Poly_Interact_Linear,Poly_Linear_interact_only,CV-5,0.143151,0.020570,0.876350,0.012133,degree,2,interaction_only,True,NA,NA,All,"Functional_Sal, Heating_Wall, Neighborhood_Grn...","-1.72, 0.31, 0.29, 0.25, -0.23, 0.22, -0.21, -..."


# Model View

In [23]:
df = pd.DataFrame(records)
df.sort_values('Log RMSE Mean', ascending=True)

,Model,Regression Type,Split,RMSE Mean,MSE Mean,R2 Mean,Log RMSE Mean,Hyperparameter 1 Name,Hyperparameter 1 Value,Hyperparameter 2 Name,Hyperparameter 2 Value,Hyperparameter 3 Name,Hyperparameter 3 Value,Variables Used,Top 10 Variables,Top 10 Coefficients
7,Ridge_Poly_Interact,Ridge_Poly_interact_only,CV-5,0.140526,0.019797,0.881043,0.011904,Alpha,7.196857,degree,2,interaction_only,True,All,"Gr Liv Area, Functional_Sal, Bldg Type_Duplex,...","0.21, -0.17, -0.12, 0.11, 0.11, 0.11, -0.11, -..."
6,Ridge_Poly_Full,Ridge_Poly_full,CV-5,0.140536,0.019807,0.881016,0.011925,Alpha,2.811769,degree,2,interaction_only,False,All,"Functional_Sal, Gr Liv Area, Neighborhood_Mead...","-0.35, 0.22, -0.13, 0.13, -0.12, -0.12, 0.12, ..."
8,Poly_Full_Linear,Poly_Linear_full,CV-5,0.142488,0.020382,0.877540,0.012086,degree,2,interaction_only,False,NA,NA,All,"Functional_Sal, Heating_Wall, Neighborhood_Grn...","-1.74, 0.30, 0.29, 0.25, 0.23, -0.21, 0.21, 0...."
9,Poly_Interact_Linear,Poly_Linear_interact_only,CV-5,0.143151,0.020570,0.876350,0.012133,degree,2,interaction_only,True,NA,NA,All,"Functional_Sal, Heating_Wall, Neighborhood_Grn...","-1.72, 0.31, 0.29, 0.25, -0.23, 0.22, -0.21, -..."
1,Ridge_All_Tuned,Ridge,CV-5,0.148614,0.022242,0.866493,0.012455,Alpha,6.866488,NA,NA,NA,NA,All,"Functional_Sal, Neighborhood_NridgHt, Gr Liv A...","-0.17, 0.16, 0.16, -0.16, -0.14, 0.14, -0.13, ..."
0,All_Features_Linear,Linear,CV-5,0.149727,0.022542,0.864659,0.012569,NA,NA,NA,NA,NA,NA,All,"Functional_Sal, Functional_Sev, Neighborhood_G...","-1.64, -0.33, 0.28, 0.28, 0.27, -0.24, 0.23, -..."
3,ElasticNet_All_Tuned,ElasticNet,CV-5,0.159918,0.025770,0.845418,0.013374,Alpha,0.1,L1 Ratio,0,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Overall...","0.15, 0.12, 0.10, 0.05, 0.05, 0.04, 0.04, -0.0..."
4,KNN_All_Tuned,KNN,CV-5,0.165188,0.027344,0.837218,0.013945,n_neighbors,7,weights,distance,metric,manhattan,All,NA (KNN has no coefficients),NA (KNN has no coefficients)
5,DecisionTree_All_Tuned,Decision Tree,CV-5,0.178749,0.032041,0.809408,0.015043,max_depth,15,min_samples_leaf,2,min_samples_split,20,All,"Overall Qual, Gr Liv Area, Year Built, Lot Are...","0.7177, 0.1481, 0.0545, 0.0339, 0.0109, 0.0049..."
2,Lasso_All_Tuned,Lasso,CV-5,0.224634,0.050538,0.699003,0.018851,Alpha,0.1,NA,NA,NA,NA,All,"Overall Qual, Gr Liv Area, Year Built, Lot Fro...","0.18, 0.07, 0.03, 0.00, -0.00, -0.00, -0.00, -..."


# Final Model Selection

In [24]:
# # Choose your best 
# best_model_name = "Ridge_Poly_Interact"
# final_model = model_library[best_model_name]

# # Display the metrics for your chosen model
# results_df = pd.DataFrame(records)
# print("=== Model metrics ===")
# print(results_df[results_df["Model"] == best_model_name][["Model", "Log RMSE Mean", "RMSE Mean", "R2 Mean"]].to_string(index=False))

In [25]:
# # Make predictions on test data
# y_test_pred = final_model.predict(X_test)

# # Clip any negative predictions (house prices can't be negative)
# y_test_pred = np.clip(y_test_pred, 1, None)

# print(f"Predictions range: ${y_test_pred.min():,.0f} - ${y_test_pred.max():,.0f}")

In [26]:
# # Build submission
# submission = pd.DataFrame({
#     "PID": test_PID,
#     "SalePrice": y_test_pred
# })

# # Preview
# print(submission.head())

# # Save to your project folder
# submission_path = "/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-regression/regression_submission.csv"
# submission.to_csv(submission_path, index=False)
# print(f"\nSaved submission to {submission_path}")

# Final Model Selection with Unlog 

In [30]:
# Choose your best model (change this name to match your best performing model)
best_model_name = "Ridge_Poly_Interact"

# Get the model from your library
final_model = model_library[best_model_name]

# Display the metrics for your chosen model
results_df = pd.DataFrame(records)
print("=== Your chosen model's metrics ===")
print(results_df[results_df["Model"] == best_model_name][["Model", "Log RMSE Mean", "RMSE Mean", "R2 Mean"]].to_string(index=False))

=== Your chosen model's metrics ===
              Model  Log RMSE Mean  RMSE Mean  R2 Mean
Ridge_Poly_Interact       0.011904   0.140526 0.881043


In [31]:
# Make predictions on test data (predictions come out in log scale)
y_test_pred_log = final_model.predict(X_test)

# Convert from log scale back to regular prices
y_test_pred = np.expm1(y_test_pred_log)

# Clip any negative predictions (just in case)
y_test_pred = np.clip(y_test_pred, 1, None)

print(f"Predictions range: ${y_test_pred.min():,.0f} - ${y_test_pred.max():,.0f}")

Predictions range: $54,414 - $536,681


In [32]:
# Build submission
submission = pd.DataFrame({
    "PID": test_PID,
    "SalePrice": y_test_pred
})

# Preview
print(submission.head())

# Save to your project folder
submission_path = "/Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-regression/regression_submission.csv"
submission.to_csv(submission_path, index=False)
print(f"\nSaved submission to {submission_path}")

         PID      SalePrice
0  907135180  122209.658186
1  528181040  214713.157322
2  528175010  219036.350646
3  531379030  188062.540566
4  923275090  121359.594844

Saved submission to /Users/jamescompagno/Library/CloudStorage/OneDrive-Personal/MSBA/GSB_544/Final_Project/gsb-544-fall-2025-regression/regression_submission.csv
